# Cars

## Download Cars Data

In [35]:
import pandas as pd

# read data from the provided URL
cars = pd.read_csv("https://www.ag.ch/app/sajato-api/api/v2/export?columns=SIS_MOFA0001D.1.PERS_WAGEN&prefix=multiple&sep=%E2%82%AC&search=&fileType=csv&dateFrom=&dateTo=", encoding="latin-1")

display(cars.head())

,bfsNr,year,month,day,locationName,locationType,PERS_WAGEN
0,19,2025,9,30,Kanton Aargau,CANTON,431268.0
1,19,2024,9,30,Kanton Aargau,CANTON,425737.0
2,19,2023,9,30,Kanton Aargau,CANTON,420053.0
3,19,2022,9,30,Kanton Aargau,CANTON,415196.0
4,19,2021,9,30,Kanton Aargau,CANTON,412121.0


In [36]:
# Convert year, month, day into a date
cars["date"] = cars.apply(lambda row: f"{row['year']}-{str(row['month']).zfill(2)}-{str(row['day']).zfill(2)}", axis=1)
cars["date"] = pd.to_datetime(cars["date"])

# datatype int for PERS_WAGEN
cars["PERS_WAGEN"] = cars["PERS_WAGEN"].astype(int)
display(cars.head())

,bfsNr,year,month,day,locationName,locationType,PERS_WAGEN,date
0,19,2025,9,30,Kanton Aargau,CANTON,431268,2025-09-30
1,19,2024,9,30,Kanton Aargau,CANTON,425737,2024-09-30
2,19,2023,9,30,Kanton Aargau,CANTON,420053,2023-09-30
3,19,2022,9,30,Kanton Aargau,CANTON,415196,2022-09-30
4,19,2021,9,30,Kanton Aargau,CANTON,412121,2021-09-30


In [37]:
# Only municipalities (and no districts or canton)
cars = cars[cars["locationType"] == "TOWNSHIP"]
display(cars.head())

,bfsNr,year,month,day,locationName,locationType,PERS_WAGEN,date
620,4001,2025,9,30,Aarau,TOWNSHIP,10780,2025-09-30
621,4001,2024,9,30,Aarau,TOWNSHIP,10717,2024-09-30
622,4001,2023,9,30,Aarau,TOWNSHIP,10647,2023-09-30
623,4001,2022,9,30,Aarau,TOWNSHIP,10556,2022-09-30
624,4001,2021,9,30,Aarau,TOWNSHIP,10618,2021-09-30


In [38]:
from rdf_notebook import query_endpoint

query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?bfsNr ?version ?versionId ?startEvent ?startDate WHERE {
    ?muni a <https://schema.ld.admin.ch/PoliticalMunicipality> ;
        schema:containedInPlace <https://ld.admin.ch/canton/19> ;
        schema:identifier ?bfsNr.
    
    ?version vl:identity ?muni;
        schema:identifier ?versionId;
        schema:validFrom ?startDate;
        vl:startEvent ?startEvent.

} ORDER BY DESC(?startDate)

"""

versions = query_endpoint("https://ld.admin.ch/query", query)

versions["startDate"] = pd.to_datetime(versions["startDate"])

# cast bfsNr to int
versions["bfsNr"] = versions["bfsNr"].astype(int)
display(versions.head(10))

,bfsNr,version,versionId,startEvent,startDate
0,4095,https://ld.admin.ch/municipality/version/16673,16673,https://ld.admin.ch/municipality/changeevent/4003,2026-01-01
1,4021,https://ld.admin.ch/municipality/version/16655,16655,https://ld.admin.ch/municipality/changeevent/3987,2024-01-01
2,4139,https://ld.admin.ch/municipality/version/16625,16625,https://ld.admin.ch/municipality/changeevent/3968,2023-01-01
3,4186,https://ld.admin.ch/municipality/version/16626,16626,https://ld.admin.ch/municipality/changeevent/3969,2023-01-01
4,4185,https://ld.admin.ch/municipality/version/16615,16615,https://ld.admin.ch/municipality/changeevent/3958,2022-01-01
5,4324,https://ld.admin.ch/municipality/version/16616,16616,https://ld.admin.ch/municipality/changeevent/3959,2022-01-01
6,4095,https://ld.admin.ch/municipality/version/16135,16135,https://ld.admin.ch/municipality/changeevent/3927,2020-01-01
7,4281,https://ld.admin.ch/municipality/version/16126,16126,https://ld.admin.ch/municipality/changeevent/3628,2019-01-01
8,4104,https://ld.admin.ch/municipality/version/16087,16087,https://ld.admin.ch/municipality/changeevent/3589,2018-01-01
9,4063,https://ld.admin.ch/municipality/version/15644,15644,https://ld.admin.ch/municipality/changeevent/3422,2014-01-01


In [39]:
# Helper function to get version based on date
def get_description(datarow):
    
    versions_filtered = versions[versions["bfsNr"] == datarow["bfsNr"]]
    
    for index, row in versions_filtered.iterrows():
        if row["startDate"] <= datarow["date"]:
            return f"{row['startEvent']}/description"
    return None

In [40]:
cars["muniDescription"] = cars.apply(get_description, axis=1)

display(cars.head())

,bfsNr,year,month,day,locationName,locationType,PERS_WAGEN,date,muniDescription
620,4001,2025,9,30,Aarau,TOWNSHIP,10780,2025-09-30,https://ld.admin.ch/municipality/changeevent/3167/description
621,4001,2024,9,30,Aarau,TOWNSHIP,10717,2024-09-30,https://ld.admin.ch/municipality/changeevent/3167/description
622,4001,2023,9,30,Aarau,TOWNSHIP,10647,2023-09-30,https://ld.admin.ch/municipality/changeevent/3167/description
623,4001,2022,9,30,Aarau,TOWNSHIP,10556,2022-09-30,https://ld.admin.ch/municipality/changeevent/3167/description
624,4001,2021,9,30,Aarau,TOWNSHIP,10618,2021-09-30,https://ld.admin.ch/municipality/changeevent/3167/description


In [41]:
# keep only relevant columns
cars = cars[["bfsNr", "date", "muniDescription", "PERS_WAGEN"]]

# Save the processed data
cars.to_csv("../data/csv/cars.csv", index=False)

## Cube for Cars

In [42]:
import pandas as pd
import yaml
import os

from pylindas.pycube import Cube
from pylindas.lindas.namespaces import SCHEMA

ENVIRONMENT = os.getenv("CI_ENVIRONMENT_NAME")

# Load data
df = pd.read_csv("../data/csv/cars.csv", 
                 encoding="utf-8")

# Load configuration
with open("../data/yaml/description.yaml", encoding="utf-8") as file:
    cube_yaml = yaml.safe_load(file)

cube = Cube(dataframe=df, cube_yaml=cube_yaml, environment=ENVIRONMENT, local=True)

cube.prepare_data()
cube.write_cube()
cube.write_observations()
cube.write_shape()

cube.serialize("../data/ttl/cars.ttl")

## Self Contained CSV for Visualization Demonstrations

In [43]:
from rdf_notebook import query_ttl

# unique muniDescriptions
descriptions = cars["muniDescription"].unique()

with open("../data/ttl/events.ttl", encoding="utf-8") as file:
    ttl_content = file.read()

mapping_dict = {}

def get_muni_string(description):
    
    query = f"""

    PREFIX vl: <https://version.link/>
    PREFIX schema: <http://schema.org/> 

    SELECT ?string WHERE {{
        <{description}> schema:name ?string .
        FILTER (lang(?string) = 'en')
    }}
    """
    
    result = query_ttl(ttl_content, query)
    return (result["string"][0])

for description in descriptions:
    mapping_dict[description] = get_muni_string(description)

In [44]:
# replace muniDescription with the string from the ttl file
cars["muniDescription"] = cars["muniDescription"].map(mapping_dict)

In [45]:
# save the updated dataframe
cars.to_csv("../data/csv/cars_strings.csv", index=False)